## E0: Setup
Load pipeline outputs, define shared helpers, configure `N_RUNS`.

In [ ]:
# ── E0: Setup ─────────────────────────────────────────────────────────────────
# Load all five pipeline_output/*.json files plus run_meta.json.
# Judge prompts and functions live in evals_core.py (search E-numbers there).
# N_RUNS controls how many times dynamic evals re-invoke LLM functions.

import os, json, itertools, statistics, random, io, base64
from collections import defaultdict, Counter

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

import pipeline_core
import evals_core

OUTPUT_DIR = pipeline_core.OUTPUT_DIR
N_RUNS     = 3

def _load(name):
    path = os.path.join(OUTPUT_DIR, f"{name}.json")
    with open(path, encoding="utf-8") as f:
        return json.load(f)

db              = _load("db")
interview_store = _load("interview_store")
global_store    = _load("global_store")
lineage         = _load("lineage")
clusters        = _load("clusters")
experiments     = _load("experiments")

try:
    run_meta = _load("run_meta")
    print("Run metadata:")
    for k, v in run_meta.items():
        print(f"  {k}: {v}")
except FileNotFoundError:
    run_meta = {}
    print("run_meta.json not found — re-run C11 in Pipeline_Execution.ipynb")

print(f"\nLoaded: {len(db)} db entries, {len(interview_store)} interviews,")
print(f"        {len(global_store['l3_codes'])} L3 codes, {len(clusters)} clusters")
print(f"\nN_RUNS = {N_RUNS}")

def jaccard(a: set, b: set) -> float:
    if not a and not b:
        return 1.0
    return len(a & b) / len(a | b)

def partition_labels(cluster_map: dict, code_list: list) -> list:
    code_to_label = {}
    for idx, (name, codes) in enumerate(cluster_map.items()):
        for c in codes:
            code_to_label[c] = idx
    return [code_to_label.get(c, -1) for c in code_list]

def fig_to_b64(fig) -> str:
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)
    return base64.b64encode(buf.read()).decode()

l3_list      = [item["code"] for item in global_store["l3_codes"]]
eval_results = {}
print("\nSetup complete.")


## E1: L1 Coding Stability (dynamic)
Re-runs `code_one_answer` N_RUNS times on every Q&A entry. Pairwise Jaccard of the code sets measures per-answer stability.

**Gate:** mean Jaccard < 0.6 flags the L1 foundation as noise-prone.

In [ ]:
# ── E1: L1 Coding Stability (dynamic) ────────────────────────────────────────
# code_one_answer defined in pipeline_core.py (search "C6").
from pipeline_core import code_one_answer

print(f"Running L1 coder {N_RUNS}x per entry ({len(db)} entries = {len(db)*N_RUNS} LLM calls)...")
runs = []
for i in range(N_RUNS):
    run = {iq_id: set(code_one_answer(e["question"], e["anonymised_answer"]))
           for iq_id, e in db.items()}
    runs.append(run)
    print(f"  Run {i+1}/{N_RUNS} done")

pairs = list(itertools.combinations(range(N_RUNS), 2))
per_answer = {iq_id: statistics.mean(jaccard(runs[i][iq_id], runs[j][iq_id]) for i, j in pairs)
              for iq_id in db}

all_scores  = list(per_answer.values())
mean_j      = statistics.mean(all_scores)
gate_passed = mean_j >= 0.6

fig, ax = plt.subplots(figsize=(7, 3.5))
color = "#4f8ef7" if gate_passed else "#f74f4f"
ax.hist(all_scores, bins=15, color=color, alpha=0.85, edgecolor="white")
ax.axvline(0.6, color="#dc2626", linestyle="--", linewidth=1.5, label="Gate (0.6)")
ax.axvline(mean_j, color="#16a34a", linestyle="-", linewidth=1.5, label=f"Mean ({mean_j:.2f})")
ax.set_xlabel("Mean pairwise Jaccard similarity")
ax.set_ylabel("Number of Q&A entries")
ax.set_title(f"E1: L1 Coding Stability ({N_RUNS} runs)")
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
plt.tight_layout()
e1_fig = fig_to_b64(fig)

worst = sorted(per_answer.items(), key=lambda x: x[1])[:5]
print(f"\nE1: mean Jaccard={mean_j:.3f}  gate >= 0.6  {'PASS' if gate_passed else 'FAIL'}")
print("5 least stable Q&A entries:")
for iq_id, score in worst:
    print(f"  {iq_id}  J={score:.2f}  Q: {db[iq_id]['question'][:60]}")

eval_results["E1"] = {
    "passed":  gate_passed,
    "figures": [e1_fig],
    "summary": f"Mean pairwise Jaccard={mean_j:.3f} (gate 0.6). {'PASS' if gate_passed else 'FAIL: L1 codes are unstable.'}",
}


## E2: L2 Lineage Integrity (static)
Audits `merged_from_l1` pointers for two failure modes:

- **Orphan references:** L1 code cited in `merged_from_l1` that never appeared in L1 output.
- **Dropped codes:** L1 codes absorbed by nothing (minority-view suppression).

In [ ]:
# ── E2: L2 Lineage Integrity (static) ────────────────────────────────────────
orphans_by_iv = {}
dropped_by_iv = {}

for interview_id, store in interview_store.items():
    source_l1  = {c for rec in db.values()
                  if rec["interview_id"] == interview_id
                  for c in rec["l1_codes"]}
    referenced = {src for l2 in store["l2_codes"] for src in l2["merged_from_l1"]}
    orphans_by_iv[interview_id[:8]] = len(referenced - source_l1)
    dropped_by_iv[interview_id[:8]] = len(source_l1 - referenced)

ids       = list(orphans_by_iv.keys())
orphan_v  = [orphans_by_iv[i] for i in ids]
dropped_v = [dropped_by_iv[i] for i in ids]

fig, ax = plt.subplots(figsize=(7, 3.5))
x = np.arange(len(ids))
ax.bar(x - 0.18, orphan_v, width=0.35, label="Orphan refs", color="#f97316", alpha=0.85)
ax.bar(x + 0.18, dropped_v, width=0.35, label="Dropped codes", color="#8b5cf6", alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(ids, fontsize=9)
ax.set_ylabel("Count")
ax.set_title("E2: L2 Lineage Integrity per interview")
ax.legend(fontsize=9)
plt.tight_layout()
e2_fig = fig_to_b64(fig)

total_orphans = sum(orphan_v)
total_dropped = sum(dropped_v)
gate_passed   = total_orphans == 0
print(f"E2: orphan refs={total_orphans}  dropped codes={total_dropped}  {'PASS' if gate_passed else 'FAIL'}")

eval_results["E2"] = {
    "passed":  gate_passed,
    "figures": [e2_fig],
    "summary": f"Orphan refs={total_orphans}, dropped={total_dropped}. {'PASS' if gate_passed else 'FAIL: hallucinated lineage pointers.'}",
}


## E2b: L2 Semantic Merge Quality (static, Claude judge)
Checks whether each L2 label faithfully represents its absorbed L1 codes.
Judge prompt defined in `evals_core.py` (search `E2b`).

**Certainty < 0.8** flags the verdict for human review.

In [ ]:
# ── E2b: L2 Semantic Merge Quality (static) ──────────────────────────────────
# Judge prompt: evals_core.MERGE_JUDGE  |  function: evals_core.run_merge_quality_check

e2b = evals_core.run_merge_quality_check(interview_store)

gate_passed = len(e2b["unfaithful"]) == 0
print(f"E2b: {len(e2b['all'])} L2 merges judged  "
      f"unfaithful={len(e2b['unfaithful'])}  low-certainty={len(e2b['low_certainty'])}")
for v in e2b["unfaithful"]:
    print(f"  FAIL  [{v['interview']}] {v['l2_code']!r}  {v['reason']}")
for v in e2b["low_certainty"]:
    print(f"  REVIEW [{v['interview']}] {v['l2_code']!r}  certainty={v.get('certainty','?')}  {v['reason']}")

eval_results["E2b"] = {
    "passed":  gate_passed,
    "figures": [],
    "data":    e2b,
    "summary": f"{len(e2b['unfaithful'])} unfaithful, {len(e2b['low_certainty'])} low-certainty. {'PASS' if gate_passed else 'FAIL: semantic merge errors.'}",
}


## E3: L3 Coverage and Suppression (static)
Measures information loss during the L2→L3 and L3→cluster funnels.
High suppression means minority views were dropped before the report.

In [ ]:
# ── E3: L3 Coverage and Suppression (static) ─────────────────────────────────
all_l2        = {l2["code"] for s in interview_store.values() for l2 in s["l2_codes"]}
referenced_l2 = {src for l3 in global_store["l3_codes"] for src in l3["merged_from_l2"]}
all_l3        = {l3["code"] for l3 in global_store["l3_codes"]}
clustered_l3  = {c for cl in clusters.values() for c in cl["l3_codes"]}

vals   = [len(all_l2), len(referenced_l2), len(all_l3), len(clustered_l3)]
labels = ["L2 total", "L2 in L3", "L3 total", "L3 in clusters"]
colors = ["#4f8ef7", "#f97316", "#4f8ef7", "#f97316"]

fig, ax = plt.subplots(figsize=(7, 3.5))
bars = ax.bar(labels, vals, color=colors, alpha=0.85, edgecolor="white")
for bar, val in zip(bars, vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            str(val), ha="center", va="bottom", fontsize=10, fontweight="bold")
ax.set_ylabel("Code count")
ax.set_title("E3: L3 Coverage Funnel (blue = total, orange = surviving)")
plt.tight_layout()
e3_fig = fig_to_b64(fig)

dropped_l2  = len(all_l2) - len(referenced_l2)
dropped_l3  = len(all_l3) - len(clustered_l3)
gate_passed = dropped_l3 == 0
print(f"E3: L2 dropped={dropped_l2}  L3 unclustered={dropped_l3}  {'PASS' if gate_passed else 'FAIL'}")

eval_results["E3"] = {
    "passed":  gate_passed,
    "figures": [e3_fig],
    "summary": f"L2 dropped={dropped_l2}, L3 unclustered={dropped_l3}. {'PASS' if gate_passed else 'FAIL: codes lost in funnel.'}",
}


## E4: C9 Clustering Stability (dynamic)
Re-clusters the same L3 codes N_RUNS times and measures pairwise ARI/NMI.

**Gate:** median ARI < 0.7 means themes are partly an artifact of LLM sampling.

In [ ]:
# ── E4: C9 Clustering Stability (dynamic) ────────────────────────────────────
# cluster_l3_codes defined in pipeline_core.py (search "C9").
from pipeline_core import cluster_l3_codes

print(f"Re-clustering {len(l3_list)} L3 codes {N_RUNS} times...")
label_runs = []
for i in range(N_RUNS):
    cmap = cluster_l3_codes(l3_list)
    label_runs.append(partition_labels(cmap, l3_list))
    print(f"  Run {i+1}/{N_RUNS}: {len(cmap)} clusters")

pairs_r = list(itertools.combinations(range(N_RUNS), 2))
aris    = [adjusted_rand_score(label_runs[i], label_runs[j]) for i, j in pairs_r]
nmis    = [normalized_mutual_info_score(label_runs[i], label_runs[j]) for i, j in pairs_r]
med_ari = statistics.median(aris)
gate_passed = med_ari >= 0.7

fig, ax = plt.subplots(figsize=(7, 3.5))
pair_labels = [f"R{i+1} vs R{j+1}" for i, j in pairs_r]
x = np.arange(len(pair_labels))
ax.bar(x - 0.18, aris, width=0.35, label="ARI", color="#4f8ef7", alpha=0.85)
ax.bar(x + 0.18, nmis, width=0.35, label="NMI", color="#22c55e", alpha=0.85)
ax.axhline(0.7, color="#dc2626", linestyle="--", linewidth=1.5, label="ARI gate (0.7)")
ax.set_xticks(x)
ax.set_xticklabels(pair_labels)
ax.set_ylim(0, 1)
ax.set_ylabel("Score")
ax.set_title(f"E4: Clustering Stability ({N_RUNS} runs)")
ax.legend(fontsize=9)
plt.tight_layout()
e4_fig = fig_to_b64(fig)

print(f"\nE4: median ARI={med_ari:.3f}  gate >= 0.7  {'PASS' if gate_passed else 'FAIL'}")

eval_results["E4"] = {
    "passed":  gate_passed,
    "figures": [e4_fig],
    "summary": f"Median ARI={med_ari:.3f} (gate 0.7). {'PASS' if gate_passed else 'FAIL: themes not stable across runs.'}",
}


## E4b: Lineage Integrity (static)
Structural audit of `clusters.json` and `lineage.json`:

- **Duplicate L3 codes** across clusters: a quote traces to two themes simultaneously.
- **Unclustered L3 codes:** silently dropped from the report.
- **Invalid IQ IDs:** lineage points to a Q&A entry that doesn't exist in `db.json`.

In [ ]:
# ── E4b: Lineage Integrity (static) ──────────────────────────────────────────
import collections

seen_l3     = collections.Counter(c for cl in clusters.values() for c in cl["l3_codes"])
all_l3s     = {l3["code"] for l3 in global_store["l3_codes"]}
duplicates  = [c for c, n in seen_l3.items() if n > 1]
unclustered = [c for c in all_l3s if c not in seen_l3]
bad_iq_ids  = [(name, e["interview_question_id"])
               for name, lin in lineage.items()
               for e in lin["l1_codes"]
               if e["interview_question_id"] not in db]

gate_passed = len(duplicates) == 0 and len(bad_iq_ids) == 0
checks = [
    ("Duplicate L3 codes",              len(duplicates) == 0,  len(duplicates)),
    ("Unclustered L3 codes",             len(unclustered) == 0, len(unclustered)),
    ("Invalid interview_question_ids",   len(bad_iq_ids) == 0,  len(bad_iq_ids)),
]

print("E4b: Lineage Integrity Audit")
print(f"  {'Check':<42} {'Status':<8} Count")
print("  " + "-"*56)
for label, passed, count in checks:
    print(f"  {label:<42} {'PASS' if passed else 'FAIL':<8} {count}")
if duplicates:
    print(f"\nDuplicate L3 codes: {duplicates}")
if unclustered:
    print(f"Unclustered: {unclustered[:5]}{' ...' if len(unclustered)>5 else ''}")

eval_results["E4b"] = {
    "passed":  gate_passed,
    "figures": [],
    "summary": f"Duplicates={len(duplicates)}, unclustered={len(unclustered)}, bad IDs={len(bad_iq_ids)}. {'PASS' if gate_passed else 'FAIL: structural lineage errors.'}",
}


## E5: Narrative Faithfulness (static, Claude judge)
Checks whether each cluster's summary narrative makes claims not supported by the source Q&A pairs.
Judge prompt defined in `evals_core.py` (search `E5`).

**Judge model:** `claude-sonnet-4-6` (different family from Haiku generator, reduces self-preference).

**Certainty < 0.8** flags the verdict for human review.

In [ ]:
# ── E5: Narrative Faithfulness (static) ──────────────────────────────────────
# Judge prompt: evals_core.FAITHFULNESS_JUDGE  |  function: evals_core.run_faithfulness_check

e5 = evals_core.run_faithfulness_check(clusters, lineage, db)

gate_passed = len(e5["unfaithful"]) == 0
print(f"E5: {len(e5['all'])} narratives judged  "
      f"unfaithful={len(e5['unfaithful'])}  review={len(e5['low_certainty'])}")
for v in e5["unfaithful"]:
    print(f"  FAIL  {v['cluster']}  certainty={v.get('certainty','?')}  {v['reason']}")
for v in e5["low_certainty"]:
    print(f"  REVIEW {v['cluster']}  certainty={v.get('certainty','?')}  {v['reason']}")

eval_results["E5"] = {
    "passed":  gate_passed,
    "figures": [],
    "data":    e5,
    "summary": f"{len(e5['unfaithful'])} unfaithful, {len(e5['low_certainty'])} low-certainty. {'PASS' if gate_passed else 'FAIL: narrative hallucination detected.'}",
}


## E5b: Sentiment Overclaim Guard (static, Claude judge)
Checks whether narratives assert positive sentiment that the evidence does not clearly support.
Judge prompt defined in `evals_core.py` (search `E5b`).

**Certainty < 0.8** flags for human review.

In [ ]:
# ── E5b: Sentiment Overclaim Guard (static) ──────────────────────────────────
# Judge prompt: evals_core.OVERCLAIM_JUDGE  |  function: evals_core.run_overclaim_check

e5b = evals_core.run_overclaim_check(clusters, lineage, db)

gate_passed = len(e5b["unsupported"]) == 0
print(f"E5b: {len(e5b['all'])} narratives checked  "
      f"unsupported={len(e5b['unsupported'])}  review={len(e5b['low_certainty'])}")
for v in e5b["unsupported"]:
    print(f"  FAIL  {v['cluster']}  certainty={v.get('certainty','?')}  {v['reason']}")
for v in e5b["low_certainty"]:
    print(f"  REVIEW {v['cluster']}  certainty={v.get('certainty','?')}  {v['reason']}")

eval_results["E5b"] = {
    "passed":  gate_passed,
    "figures": [],
    "data":    e5b,
    "summary": f"{len(e5b['unsupported'])} unsupported, {len(e5b['low_certainty'])} low-certainty. {'PASS' if gate_passed else 'FAIL: sentiment overclaim detected.'}",
}


## E6: Metamorphic Invariance (dynamic)

**(a) Paraphrase invariance at C6.** Rewrites sample answers in different words and re-codes them. Jaccard < 0.5 means the coder responds to phrasing rather than meaning.

**(b) Order invariance at C9 (Lost-in-the-Middle).** Shuffles the L3 code list and re-clusters. Low ARI means themes depend on input order, not just the data.

Paraphrase function defined in `evals_core.py` (search `E6`).

In [ ]:
# ── E6: Metamorphic Invariance (dynamic) ─────────────────────────────────────
from pipeline_core import code_one_answer, cluster_l3_codes

# (a) Paraphrase invariance on first 5 entries
sample_ids  = list(db.keys())[:5]
para_scores = []
print("E6a: Paraphrase invariance (sample 5 entries)...")
for iq_id in sample_ids:
    e      = db[iq_id]
    base   = set(code_one_answer(e["question"], e["anonymised_answer"]))
    para   = evals_core.paraphrase(e["anonymised_answer"])
    para_c = set(code_one_answer(e["question"], para))
    score  = jaccard(base, para_c)
    para_scores.append(score)
    print(f"  {iq_id}  J={score:.2f}")

mean_para  = statistics.mean(para_scores) if para_scores else 0.0
para_pass  = mean_para >= 0.5

# (b) Order invariance
print("\nE6b: Order invariance...")
shuffled   = l3_list[:]
random.shuffle(shuffled)
cmap_orig  = cluster_l3_codes(l3_list)
cmap_shuf  = cluster_l3_codes(shuffled)
part_o     = partition_labels(cmap_orig, l3_list)
part_s     = partition_labels(cmap_shuf, l3_list)
order_ari  = adjusted_rand_score(part_o, part_s)
order_pass = order_ari >= 0.7
print(f"  Order-invariance ARI={order_ari:.3f}  {'PASS' if order_pass else 'FAIL'}")

fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
axes[0].bar(range(len(para_scores)), para_scores,
            color=["#4f8ef7" if s >= 0.5 else "#f74f4f" for s in para_scores], alpha=0.85)
axes[0].axhline(0.5, color="#dc2626", linestyle="--", linewidth=1.5, label="Gate (0.5)")
axes[0].set_ylim(0, 1)
axes[0].set_xticks(range(len(sample_ids)))
axes[0].set_xticklabels([i[:8] for i in sample_ids], rotation=20, fontsize=8)
axes[0].set_title("E6a: Paraphrase Jaccard")
axes[0].legend(fontsize=8)
axes[1].bar(["Order ARI"], [order_ari],
            color="#4f8ef7" if order_pass else "#f74f4f", alpha=0.85)
axes[1].axhline(0.7, color="#dc2626", linestyle="--", linewidth=1.5, label="Gate (0.7)")
axes[1].set_ylim(0, 1)
axes[1].set_title("E6b: Order-Invariance ARI")
axes[1].legend(fontsize=8)
plt.tight_layout()
e6_fig = fig_to_b64(fig)

gate_passed = para_pass and order_pass
print(f"\nE6: para={mean_para:.3f} {'PASS' if para_pass else 'FAIL'}  order={order_ari:.3f} {'PASS' if order_pass else 'FAIL'}")

eval_results["E6"] = {
    "passed":  gate_passed,
    "figures": [e6_fig],
    "summary": f"Paraphrase Jaccard={mean_para:.3f} (gate 0.5), order ARI={order_ari:.3f} (gate 0.7). {'PASS' if gate_passed else 'FAIL.'}",
}


## E7: Leakage / Re-identification Probe (static, Claude adversary)
Can a separate LLM read the anonymised report and, given a synthetic 5-person roster, guess who said which paraphrased quote? Random baseline = 20%.
Probe function defined in `evals_core.py` (search `E7`).

**Instructions:** Edit `SYNTHETIC_ROSTER` in the first cell below, then run the second cell.

In [ ]:
# ── E7a: Synthetic roster — EDIT THIS CELL before running E7b ─────────────────
# Provide 5 fictional employees with plausible but non-real descriptions.

SYNTHETIC_ROSTER = [
    {"name": "Alex",    "role": "Senior Engineer, 4 years tenure"},
    {"name": "Jordan",  "role": "Customer Success Manager, 2 years tenure"},
    {"name": "Morgan",  "role": "Data Analyst, 1 year tenure"},
    {"name": "Sam",     "role": "Product Manager, 3 years tenure"},
    {"name": "Taylor",  "role": "HR Business Partner, 5 years tenure"},
]

print("Roster configured:")
for p in SYNTHETIC_ROSTER:
    print(f"  {p['name']:10}  {p['role']}")
print("\nEdit SYNTHETIC_ROSTER if needed, then run E7b below.")


In [ ]:
# ── E7b: Re-identification probe ─────────────────────────────────────────────
# Probe function: evals_core.run_reidentification_probe
import re as _re

report_path = os.path.join(pipeline_core.OUTPUT_DIR, "report.html")
try:
    with open(report_path, encoding="utf-8") as f:
        report_html = f.read()
    report_text = _re.sub(r"<[^>]+>", " ", report_html)
    report_text = _re.sub(r"\s+", " ", report_text).strip()
except FileNotFoundError:
    report_text = "(report.html not found — run C13 in Pipeline_Execution.ipynb first)"

e7 = evals_core.run_reidentification_probe(report_text, SYNTHETIC_ROSTER)

baseline = 1 / len(SYNTHETIC_ROSTER)
print(f"E7: {len(e7['guesses'])} quotes probed  high-conf={len(e7['high_conf'])}  baseline={baseline:.0%}")
for g in e7["guesses"]:
    flag = "  ** HIGH CONF **" if g.get("confidence", 0) >= 0.7 else ""
    print(f"  [{g.get('guess','?')} {g.get('confidence','?')}]{flag}  {g.get('quote_preview','?')}")

gate_passed = len(e7["high_conf"]) == 0
eval_results["E7"] = {
    "passed":  gate_passed,
    "figures": [],
    "data":    e7,
    "summary": f"{len(e7['guesses'])} quotes probed, {len(e7['high_conf'])} high-conf guesses. {'PASS' if gate_passed else 'REVIEW: anonymisation may be insufficient.'}",
}


## Eval Report
Writes `pipeline_output/eval_report.html` (committed to git).

In [ ]:
# ── Final: Write eval_report.html ────────────────────────────────────────────
import html as _html, webbrowser

def H(s): return _html.escape(str(s))

EVAL_CSS = """
*,*::before,*::after{box-sizing:border-box;margin:0;padding:0}
body{font-family:-apple-system,BlinkMacSystemFont,'Segoe UI',Helvetica,sans-serif;
     background:#fff;color:#111;font-size:15px;line-height:1.6}
.hdr{display:flex;align-items:center;gap:14px;padding:14px 40px;
     border-bottom:1px solid #e5e5e5;background:#fff}
.logo{width:36px;height:36px;background:#000;border-radius:8px;color:#fff;
      font-weight:800;font-size:13px;display:flex;align-items:center;justify-content:center}
.hdr-t{font-size:16px;font-weight:700}
.hdr-m{font-size:12px;color:#999}
.wrap{max-width:820px;margin:0 auto;padding:36px 24px 60px}
.sec{margin-bottom:28px;border:1px solid #e8e8e8;border-radius:10px;overflow:hidden}
.sec-hdr{display:flex;align-items:center;gap:10px;padding:12px 18px;
         border-bottom:1px solid #e8e8e8;background:#fafafa}
.pass{background:#dcfce7;color:#15803d;border-radius:20px;padding:2px 10px;
      font-size:12px;font-weight:700}
.fail{background:#fee2e2;color:#dc2626;border-radius:20px;padding:2px 10px;
      font-size:12px;font-weight:700}
.warn{background:#fef3c7;color:#b45309;border-radius:20px;padding:2px 10px;
      font-size:12px;font-weight:700}
.sec-title{font-size:14px;font-weight:700}
.sec-sub{font-size:11px;color:#aaa;margin-left:auto}
.sec-body{padding:14px 18px}
.sec-body p{font-size:13px;color:#555;margin-bottom:8px}
.fig img{max-width:100%;border-radius:6px;margin-top:8px}
.footer{text-align:center;font-size:12px;color:#ccc;padding:28px 0 8px}
"""

def _sec(eid, title, dyn=""):
    r   = eval_results.get(eid, {})
    ok  = r.get("passed")
    badge = ('<span class="pass">PASS</span>' if ok is True
             else '<span class="fail">FAIL</span>' if ok is False
             else '<span class="warn">NOT RUN</span>')
    sub  = f'<span class="sec-sub">{H(dyn)}</span>' if dyn else ""
    summ = H(r.get("summary", "Not run."))
    figs = "".join(
        f'<div class="fig"><img src="data:image/png;base64,{fig}" alt="{H(eid)}"></div>'
        for fig in r.get("figures", [])
    )
    return (
        f'<div class="sec"><div class="sec-hdr">{badge}'
        f'<span class="sec-title">{H(eid)}: {H(title)}</span>{sub}</div>'
        f'<div class="sec-body"><p>{summ}</p>{figs}</div></div>'
    )

meta_str   = " | ".join(f"{k}: {v}" for k, v in run_meta.items()) if run_meta else "no run_meta"
all_passed = all(r.get("passed") is True for r in eval_results.values())
overall    = '<span class="pass">ALL PASS</span>' if all_passed else '<span class="fail">ISSUES FOUND</span>'

body = (
    _sec("E1",  "L1 Coding Stability",          "dynamic")
    + _sec("E2",  "L2 Lineage Integrity",        "static")
    + _sec("E2b", "L2 Semantic Merge Quality",   "static, judge")
    + _sec("E3",  "L3 Coverage and Suppression", "static")
    + _sec("E4",  "C9 Clustering Stability",     "dynamic")
    + _sec("E4b", "Lineage Integrity",            "static")
    + _sec("E5",  "Narrative Faithfulness",       "static, judge")
    + _sec("E5b", "Sentiment Overclaim",          "static, judge")
    + _sec("E6",  "Metamorphic Invariance",       "dynamic")
    + _sec("E7",  "Leakage / Re-identification",  "static, adversary")
)

full_html = f"""<!DOCTYPE html>
<html lang="en"><head>
<meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Spradley: Eval Report</title>
<style>{EVAL_CSS}</style></head><body>
<header class="hdr"><div class="logo">SP</div><div>
<div class="hdr-t">Pipeline Eval Report</div>
<div class="hdr-m">{H(meta_str)}</div></div>
<div style="margin-left:auto">{overall}</div></header>
<div class="wrap">{body}
<div class="footer">Spradley &middot; app.spradley.io</div></div>
</body></html>"""

out_path = os.path.join(pipeline_core.OUTPUT_DIR, "eval_report.html")
os.makedirs(pipeline_core.OUTPUT_DIR, exist_ok=True)
with open(out_path, "w", encoding="utf-8") as f:
    f.write(full_html)

abs_path = os.path.abspath(out_path)
print(f"Written: {abs_path}")
webbrowser.open("file:///" + abs_path.replace(os.sep, "/"))
